# Team Model (goals)

The team model fitted to goals rather than to expected goals: bpl's extended
Dixon-Coles, `--team-model extended`. It is not the default any more, so every
fit here asks for it by name. `team_model_xg.ipynb` is the same territory for
the model that is.

In [ ]:
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from airsenal.db.queries.fixtures import get_fixtures_for_gameweeks
from airsenal.db.queries.gameweeks import next_gameweek
from airsenal.db.queries.teams import get_teams_for_season
from airsenal.db.session import get_session
from airsenal.game.season import CURRENT_SEASON
from airsenal.prediction.team_models import build_team_model
from airsenal.prediction.team_models.fitting import get_fitted_team_model


## Fit Model

In [ ]:
session = get_session()
CURRENT_TEAMS = get_teams_for_season(CURRENT_SEASON, session)
NEXT_GAMEWEEK = next_gameweek()


def fit_goals_model(gameweek, season=CURRENT_SEASON):
    """The Dixon-Coles fitted to goals - `xg` is the default now, so ask by name."""
    return get_fitted_team_model(
        gameweek, season, session, model=build_team_model("extended")
    )


model_team = fit_goals_model(NEXT_GAMEWEEK)

In [ ]:
# Best first, by what a team does to a match, so that every view below reads in
# the same order as the xG notebook's. bpl's parameters are additive in the log
# rate - expected goals go as exp(attack - defence + home advantage) - so a
# bigger defence concedes fewer, and the two add up where the xG model's divide.
attack_mean = model_team.model.attack.mean(axis=0)
defence_mean = model_team.model.defence.mean(axis=0)
team_index = {team: idx for idx, team in enumerate(model_team.teams)}
RANKED_TEAMS = sorted(
    CURRENT_TEAMS,
    key=lambda team: -(attack_mean[team_index[team]] + defence_mean[team_index[team]]),
)
current_idx = {team: team_index[team] for team in RANKED_TEAMS}
# One colour per team, so that the views below can be read against each other.
TEAM_COLOUR = dict(zip(RANKED_TEAMS, plt.get_cmap("tab20").colors, strict=False))


def label_points(axis, x, y, labels, fontsize=11):
    """Annotate points that sit on top of each other.

    Twenty teams land in a tight cluster, and labelling every one of them the
    same way up and to the right makes the middle of it unreadable, so the
    offsets cycle instead. Some of them still touch; nothing here solves for a
    layout.
    """
    # Cycled by where a point sits left to right rather than by the order the
    # labels arrive in, so that two teams side by side never get the same one.
    offsets = [(8, 5), (8, -12), (-30, 5), (-30, -12)]
    place = {point: order for order, point in enumerate(np.argsort(np.asarray(x)))}
    for i, (x_i, y_i, label) in enumerate(zip(x, y, labels, strict=True)):
        axis.annotate(
            label,
            (x_i, y_i),
            textcoords="offset points",
            xytext=offsets[place[i] % len(offsets)],
            fontsize=fontsize,
        )

## Match Outcome Predictions

In [ ]:
fixtures = get_fixtures_for_gameweeks([NEXT_GAMEWEEK], CURRENT_SEASON)

nrow = int(np.ceil(len(fixtures) / 2))
fig, ax = plt.subplots(nrow, 2, figsize=(8, 2.2 * nrow), sharey=True)
ax = ax.flatten()
max_goals = 10

for i, f in enumerate(fixtures):
    prob_score_h = [
        model_team.predict_score_n_proba(n, f.home_team, f.away_team)[0]
        for n in range(max_goals)
    ]
    exp_h = sum([n * prob_score_h[n] for n in range(max_goals)]) / sum(prob_score_h)

    prob_score_a = [
        model_team.predict_score_n_proba(n, f.away_team, f.home_team, home=False)[0]
        for n in range(max_goals)
    ]
    exp_a = sum([n * prob_score_a[n] for n in range(max_goals)]) / sum(prob_score_a)

    outcome_proba = model_team.predict_outcome_proba(f.home_team, f.away_team)
    ax[i].bar(
        range(max_goals),
        prob_score_h,
        facecolor="none",
        edgecolor="b",
        label=f.home_team,
    )
    ax[i].bar(
        range(max_goals),
        prob_score_a,
        facecolor="none",
        edgecolor="r",
        label=f.away_team,
    )

    ax[i].set_xlim([-1, max_goals])
    bs = r"$\bf{"
    be = r"}$"
    ax[i].set_title(
        f"{bs}{f.home_team}{be} {bs}vs.{be} {bs}{f.away_team}{be}\n"
        f"{exp_h:.2f} goals vs. {exp_a:.2f} goals\n"
        f"Home win: {outcome_proba['home_win'][0]:.2f}, "
        f"Draw: {outcome_proba['draw'][0]:.2f}, "
        f"Away win: {outcome_proba['away_win'][0]:.2f}",
        fontsize=10,
    )
    ax[i].set_ylabel("Probability")
    ax[i].set_xlabel("Goals")
    ax[i].legend()

fig.suptitle(
    f"Predicted Match Outcomes for GW{NEXT_GAMEWEEK}, Season {CURRENT_SEASON}",
    weight="bold",
)
fig.tight_layout()

## Team Strengths / Model Parameters

In [ ]:
a_mean = model_team.model.attack.mean(axis=0)
b_mean = model_team.model.defence.mean(axis=0)

a_conf = np.abs(
    np.quantile(model_team.model.attack, [0.5 - 1 / 3, 0.5 + 1 / 3], axis=0) - a_mean
)
b_conf = np.abs(
    np.quantile(model_team.model.defence, [0.5 - 1 / 3, 0.5 + 1 / 3], axis=0) - b_mean
)

fig, ax = plt.subplots(1, 1, figsize=(8, 8))
ax.set_aspect("equal")
select_idx = jnp.array(list(current_idx.values()), dtype=int)
plt.errorbar(
    a_mean[select_idx],
    b_mean[select_idx],
    xerr=a_conf[:, select_idx],
    yerr=b_conf[:, select_idx],
    marker="o",
    markersize=10,
    linestyle="",
    linewidth=0.5,
)
plt.xlabel("attack", fontsize=14)
plt.ylabel("defence", fontsize=14)

label_points(ax, a_mean[select_idx], b_mean[select_idx], list(current_idx))

In [ ]:
plt.figure(figsize=(10, 5))
sns.violinplot(model_team.model.attack[:, list(current_idx.values())])
plt.xticks(
    ticks=np.arange(len(current_idx)), labels=list(current_idx.keys()), rotation=90
)
plt.title("Attack")
plt.axhline(0, color="k")

In [ ]:
plt.figure(figsize=(10, 5))
sns.violinplot(model_team.model.defence[:, list(current_idx.values())])
plt.xticks(
    ticks=np.arange(len(current_idx)), labels=list(current_idx.keys()), rotation=90
)
plt.title("Defence")
plt.axhline(0, color="k")

In [ ]:
plt.figure(figsize=(10, 5))
sns.violinplot(model_team.model.home_advantage[:, list(current_idx.values())])
plt.xticks(
    ticks=np.arange(len(current_idx)), labels=list(current_idx.keys()), rotation=90
)
plt.title("Home Advantage")
plt.axhline(0, color="k")

In [ ]:
plt.figure()
for i in range(model_team.model.attack_coefficients.shape[1]):
    sns.kdeplot(model_team.model.attack_coefficients[:, i])
plt.title("Attack Coefficients (Covariates)")

plt.figure()
for i in range(model_team.model.defence_coefficients.shape[1]):
    sns.kdeplot(model_team.model.defence_coefficients[:, i])
plt.title("Defence Coefficients (Covariates)")

In [ ]:
beta_a_mean = model_team.model.attack_coefficients.mean(axis=0)
beta_b_mean = model_team.model.defence_coefficients.mean(axis=0)

beta_a_conf95 = np.abs(
    np.quantile(model_team.model.attack_coefficients, [0.025, 0.975], axis=0) - beta_a_mean
)
beta_b_conf95 = np.abs(
    np.quantile(model_team.model.defence_coefficients, [0.025, 0.975], axis=0) - beta_b_mean
)
beta_a_conf80 = np.abs(
    np.quantile(model_team.model.attack_coefficients, [0.1, 0.9], axis=0) - beta_a_mean
)
beta_b_conf80 = np.abs(
    np.quantile(model_team.model.defence_coefficients, [0.1, 0.9], axis=0) - beta_b_mean
)

fig, ax = plt.subplots(1, 1, figsize=(7, 7))
ax.set_aspect("equal")
plt.errorbar(
    beta_a_mean,
    beta_b_mean,
    xerr=beta_a_conf80,
    yerr=beta_b_conf80,
    marker="o",
    markersize=10,
    linestyle="",
    linewidth=0.5,
)
plt.xlabel("beta_a", fontsize=14)
plt.ylabel("beta_b", fontsize=14)
plt.title("FIFA Ratings")

for idx, feat in enumerate(["att", "mid", "defn", "ovr"]):
    ax.annotate(feat, (beta_a_mean[idx] - 0.03, beta_b_mean[idx] + 0.02), fontsize=12)

xlim = ax.get_xlim()
ylim = ax.get_ylim()
plt.plot([0, 0], ylim, color="k", linewidth=0.75)
plt.plot(xlim, [0, 0], color="k", linewidth=0.75)
plt.xlim(xlim)
plt.ylim(ylim)

In [ ]:
sns.kdeplot(model_team.model.rho)
plt.title("rho")
print(f"Rho: {model_team.model.rho.mean():.2f}")

### Change since start of season

In [ ]:
model_team = fit_goals_model(1)
select_idx = jnp.array(
    [list(model_team.model.teams).index(t) for t in RANKED_TEAMS], dtype=int
)
a_mean_1 = model_team.model.attack.mean(axis=0)[select_idx]
b_mean_1 = model_team.model.defence.mean(axis=0)[select_idx]

model_team = fit_goals_model(NEXT_GAMEWEEK)
select_idx = jnp.array(
    [list(model_team.model.teams).index(t) for t in RANKED_TEAMS], dtype=int
)
a_mean_2 = model_team.model.attack.mean(axis=0)[select_idx]
b_mean_2 = model_team.model.defence.mean(axis=0)[select_idx]

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(7, 7))
ax.set_aspect("equal")

for a_1, b_1, a_2, b_2 in zip(a_mean_1, b_mean_1, a_mean_2, b_mean_2, strict=True):
    plt.arrow(
        a_1,
        b_1,
        a_2 - a_1,
        b_2 - b_1,
        width=0.005,
        length_includes_head=True,
    )
plt.xlabel("attack", fontsize=14)
plt.ylabel("defence", fontsize=14)

label_points(ax, a_mean_2, b_mean_2, RANKED_TEAMS)

plt.title(f"GW{NEXT_GAMEWEEK} vs. GW1")
plt.show()

delta_a = a_mean_2 - a_mean_1
delta_b = b_mean_2 - b_mean_1

print("Attack strength change since start of season:")
display(pd.Series(delta_a, index=RANKED_TEAMS).sort_values(ascending=False))

print("Defence strength change since start of season:")
display(pd.Series(delta_b, index=RANKED_TEAMS).sort_values(ascending=False))

print("Overall magnitude of team strength change since start of season:")
surprise = np.sqrt(delta_a**2 + delta_b**2)
display(pd.Series(surprise, index=RANKED_TEAMS).sort_values(ascending=False))

### Change since last gameweek 

In [ ]:
model_team = fit_goals_model(NEXT_GAMEWEEK - 1)
select_idx = jnp.array(
    [list(model_team.model.teams).index(t) for t in RANKED_TEAMS], dtype=int
)
a_mean_1 = model_team.model.attack.mean(axis=0)[select_idx]
b_mean_1 = model_team.model.defence.mean(axis=0)[select_idx]

model_team = fit_goals_model(NEXT_GAMEWEEK)
select_idx = jnp.array(
    [list(model_team.model.teams).index(t) for t in RANKED_TEAMS], dtype=int
)
a_mean_2 = model_team.model.attack.mean(axis=0)[select_idx]
b_mean_2 = model_team.model.defence.mean(axis=0)[select_idx]

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(7, 7))
ax.set_aspect("equal")
plt.axhline(0, color="k", linewidth=0.5)
plt.axvline(0, color="k", linewidth=0.5)

delta_a = a_mean_2 - a_mean_1
delta_b = b_mean_2 - b_mean_1

for da, db, t in zip(delta_a, delta_b, RANKED_TEAMS, strict=True):
    plt.plot(
        da,
        db,
        marker="none",
    )
    plt.text(
        da,
        db,
        t,
        horizontalalignment="center",
        verticalalignment="center",
        fontsize=10,
    )

plt.xlabel(r"$\Delta$(attack)", fontsize=11)
plt.ylabel(r"$\Delta$(defence)", fontsize=11)
plt.title(f"GW{NEXT_GAMEWEEK} vs. GW{NEXT_GAMEWEEK - 1}")
plt.show()

surprise = np.sqrt(delta_a**2 + delta_b**2)
print(
    f"Magnitude of team strength updates GW{NEXT_GAMEWEEK} vs. GW{NEXT_GAMEWEEK - 1}:"
)
pd.Series(surprise, index=RANKED_TEAMS).sort_values(ascending=False)

### Changes during season (slow)

In [ ]:
a_mean = np.full((len(RANKED_TEAMS), NEXT_GAMEWEEK), np.nan)
b_mean = np.full((len(RANKED_TEAMS), NEXT_GAMEWEEK), np.nan)

for gw in range(1, NEXT_GAMEWEEK + 1):
    model_team = fit_goals_model(gw)
    select_idx = jnp.array(
        [list(model_team.model.teams).index(t) for t in RANKED_TEAMS], dtype=int
    )
    a_mean[:, gw - 1] = model_team.model.attack.mean(axis=0)[select_idx]
    b_mean[:, gw - 1] = model_team.model.defence.mean(axis=0)[select_idx]

In [ ]:
fig = plt.figure(figsize=(8, 8))
for a_team, b_team, team in zip(a_mean, b_mean, RANKED_TEAMS, strict=True):
    plt.plot(a_team, b_team, marker="none", color=TEAM_COLOUR[team], label=team)
label_points(plt.gca(), a_mean[:, -1], b_mean[:, -1], RANKED_TEAMS)
plt.xlabel("Attack")
plt.ylabel("Defence")
plt.axis("equal")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 6), sharex=True)
for idx, team in enumerate(RANKED_TEAMS):
    ax[0].plot(
        np.arange(1, NEXT_GAMEWEEK + 1),
        a_mean[idx],
        marker="o",
        color=TEAM_COLOUR[team],
        label=team,
    )
    ax[1].plot(
        np.arange(1, NEXT_GAMEWEEK + 1),
        b_mean[idx],
        marker="o",
        color=TEAM_COLOUR[team],
    )
ax[0].set_xticks(np.arange(1, NEXT_GAMEWEEK + 1))
ax[0].set_xlabel("Gameweek")
ax[1].set_xlabel("Gameweek")
ax[0].set_title("Attack")
ax[1].set_title("Defence")
fig.legend(
    loc="outside lower center",
    ncol=int(np.ceil(len(RANKED_TEAMS) / 2)),
    bbox_to_anchor=(0.5, -0.15),
)